In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [2]:
df = pd.read_csv("food_waste_cleaned.csv")
df.head()

,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465
1,4,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,beginner,meat,53.008323
2,5,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,beginner,meat,48.621527
3,6,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,intermediate,meat,44.156984
4,7,2023-03-02,241,18,16.863506,79.285919,3,0,11.834878,intermediate,dairy,27.393670


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 747 entries, 0 to 746
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                747 non-null    int64  
 1   date              747 non-null    str    
 2   meals_served      747 non-null    int64  
 3   kitchen_staff     747 non-null    int64  
 4   temperature_C     747 non-null    float64
 5   humidity_percent  747 non-null    float64
 6   day_of_week       747 non-null    int64  
 7   special_event     747 non-null    int64  
 8   past_waste_kg     747 non-null    float64
 9   staff_experience  747 non-null    str    
 10  waste_category    747 non-null    str    
 11  food_waste_kg     747 non-null    float64
dtypes: float64(4), int64(5), str(3)
memory usage: 88.5 KB


In [4]:
df.dtypes

ID                    int64
date                    str
meals_served          int64
kitchen_staff         int64
temperature_C       float64
humidity_percent    float64
day_of_week           int64
special_event         int64
past_waste_kg       float64
staff_experience        str
waste_category          str
food_waste_kg       float64
dtype: object

In [5]:
df['date'] = pd.to_datetime(df['date']) 

In [6]:
df.dtypes

ID                           int64
date                datetime64[us]
meals_served                 int64
kitchen_staff                int64
temperature_C              float64
humidity_percent           float64
day_of_week                  int64
special_event                int64
past_waste_kg              float64
staff_experience               str
waste_category                 str
food_waste_kg              float64
dtype: object

In [7]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int) 

In [8]:
df = df.drop(columns=['ID', 'date'])

In [9]:
df.head()

,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,year,month,day,week_of_year,is_weekend
0,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465,2022,12,19,51,0
1,148,16,27.714300,69.046113,1,0,41.184305,beginner,meat,53.008323,2022,2,1,5,0
2,157,19,19.173902,46.292823,6,0,41.543492,beginner,meat,48.621527,2023,3,19,11,1
3,297,10,26.375233,79.741064,0,0,26.525097,intermediate,meat,44.156984,2022,7,18,29,0
4,241,18,16.863506,79.285919,3,0,11.834878,intermediate,dairy,27.393670,2023,3,2,9,0


In [10]:
x = df.drop(columns=['food_waste_kg'])
y = df['food_waste_kg']

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (747, 14)
y shape: (747,)


In [11]:
categorical_features = ['staff_experience','waste_category']

numerical_features = [
    'meals_served',
    'kitchen_staff',
    'temperature_C',
    'humidity_percent',
    'day_of_week',
    'special_event',
    'past_waste_kg',
    'year',
    'month',
    'day',
    'week_of_year',
    'is_weekend'
]

In [12]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=42)
print("Training samples:", x_train.shape[0])
print("Testing samples:", x_test.shape[0])

Training samples: 597
Testing samples: 150


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features), 
                  ('num', 'passthrough', numerical_features)]) 

In [14]:
rf_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))])
rf_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['meals_served','kitchen_staff','temperature_C',...,'day','week_of_year', 'is_weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This s

In [15]:
y_pred_rf = rf_model.predict(x_test)
print(y_pred_rf[:10])

[ 55.09749224 197.0058904   35.41142055  25.65647666  58.69283122
  25.80780788  27.89255588  41.89244326  42.58459165 203.62120922]


In [16]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
print("Random Forest Results")
print("---------------------")
print(f"MAE  : {mae_rf:.3f}")
print(f"RMSE : {rmse_rf:.3f}")
print(f"R2 Score   : {r2_rf:.3f}")


Random Forest Results
---------------------
MAE  : 5.603
RMSE : 11.947
R2 Score   : 0.904


In [17]:
gb_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=200,learning_rate=0.05,max_depth=3,random_state=42))]) 

gb_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['meals_served','kitchen_staff','temperature_C',...,'day','week_of_year', 'is_weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This s

In [18]:
y_pred_gb = gb_model.predict(x_test)

print(y_pred_gb[:10])

[ 52.73852237 190.04884983  35.13801863  27.24942985  59.19942583
  28.20200286  37.07875788  42.66605807  43.18066415 210.40103817]


In [19]:
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)
print("Gradient Boosting Results")
print("="*60)
print(f"MAE  : {mae_gb:.3f}")
print(f"RMSE : {rmse_gb:.3f}")
print(f"R2 Score : {r2_gb:.3f}")

Gradient Boosting Results
MAE  : 5.715
RMSE : 11.196
R2 Score : 0.916


In [20]:
linear_preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), numerical_features),
                  ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)]) 
lr_model = Pipeline(
    steps=[('preprocessor', linear_preprocessor),('model', LinearRegression())])
lr_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['meals_served','kitchen_staff','temperature_C',...,'day','week_of_year', 'is_weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This s

In [21]:
y_pred_lr=lr_model.predict(x_test)
print(y_pred_lr[:10])

[ 58.07481869 192.33353798  37.16449473  27.23042421  60.93064652
  28.93764197  37.62185504  42.70781148  43.18653613 223.1793922 ]


In [22]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
print("Linear Regression Results")
print("="*60)
print(f"MAE  : {mae_lr:.3f}")
print(f"RMSE : {rmse_lr:.3f}")
print(f"R2 Score : {r2_lr:.3f}")

Linear Regression Results
MAE  : 6.237
RMSE : 11.818
R2 Score : 0.906


In [23]:
rf_train_pred = rf_model.predict(x_train)
gb_train_pred = gb_model.predict(x_train)
lr_train_pred = lr_model.predict(x_train)

rf_train_r2 = r2_score(y_train, rf_train_pred)
gb_train_r2 = r2_score(y_train, gb_train_pred)
lr_train_r2 = r2_score(y_train, lr_train_pred)

print("Training vs Testing R2")
print("="*60)

print(f"Linear Regression  - Train: {lr_train_r2:.3f} | Test: {r2_lr:.3f}")
print(f"Random Forest      - Train: {rf_train_r2:.3f} | Test: {r2_rf:.3f}")
print(f"Gradient Boosting  - Train: {gb_train_r2:.3f} | Test: {r2_gb:.3f}")

Training vs Testing R2
Linear Regression  - Train: 0.836 | Test: 0.906
Random Forest      - Train: 0.988 | Test: 0.904
Gradient Boosting  - Train: 0.982 | Test: 0.916


In [24]:
param_grid = {
    'model__n_estimators': [100, 150, 200, 250],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

In [25]:
gb_search = RandomizedSearchCV(
    gb_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

In [26]:
gb_search.fit(x_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly

In [27]:
print("Best Parameters:")
print(gb_search.best_params_)

Best Parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_depth': 2, 'model__learning_rate': 0.05}


In [28]:
tuned_gb_model = gb_search.best_estimator_

In [29]:
y_pred_tuned_gb = tuned_gb_model.predict(x_test)

In [30]:
mae_tuned_gb = mean_absolute_error(y_test, y_pred_tuned_gb)
rmse_tuned_gb = np.sqrt(mean_squared_error(y_test, y_pred_tuned_gb))
r2_tuned_gb = r2_score(y_test, y_pred_tuned_gb)
print("Tuned Gradient Boosting Results")
print("="*60)
print(f"MAE  : {mae_tuned_gb:.3f}")
print(f"RMSE : {rmse_tuned_gb:.3f}")
print(f"R2 Score : {r2_tuned_gb:.3f}")

Tuned Gradient Boosting Results
MAE  : 5.654
RMSE : 11.537
R2 Score : 0.911


In [31]:
tuned_gb_train_pred = tuned_gb_model.predict(x_train)
tuned_gb_train_r2 = r2_score(y_train, tuned_gb_train_pred)
print("Tuned Gradient Boosting")
print(f"Training R2 : {tuned_gb_train_r2:.3f}")
print(f"Testing R2  : {r2_tuned_gb:.3f}")
print(f"R2 Gap      : {tuned_gb_train_r2 - r2_tuned_gb:.3f}")

Tuned Gradient Boosting
Training R2 : 0.968
Testing R2  : 0.911
R2 Gap      : 0.057


In [32]:
time_df = pd.read_csv("food_waste_cleaned.csv")

time_df['date'] = pd.to_datetime(time_df['date'])

time_df['year'] = time_df['date'].dt.year
time_df['month'] = time_df['date'].dt.month
time_df['day'] = time_df['date'].dt.day
time_df['week_of_year'] = time_df['date'].dt.isocalendar().week.astype(int)
time_df['is_weekend'] = (time_df['date'].dt.dayofweek >= 5).astype(int)

time_df = time_df.sort_values('date').reset_index(drop=True)

print(time_df[['date', 'food_waste_kg']].head())
print(time_df[['date', 'food_waste_kg']].tail())

        date  food_waste_kg
0 2022-01-01      39.901110
1 2022-01-01      39.901110
2 2022-01-02      53.203375
3 2022-01-02      53.203375
4 2022-01-03      61.375392
          date  food_waste_kg
742 2024-09-19     185.694333
743 2024-09-21      43.741845
744 2024-09-24      29.354235
745 2024-09-25      38.971427
746 2024-09-26      16.806292


In [33]:
time_df = time_df.drop(columns=['ID', 'date'])

In [34]:
x_time = time_df.drop(columns=['food_waste_kg'])
y_time = time_df['food_waste_kg']

In [35]:
split_index = int(len(time_df) * 0.80)

x_train_time = x_time.iloc[:split_index]
x_test_time = x_time.iloc[split_index:]

y_train_time = y_time.iloc[:split_index]
y_test_time = y_time.iloc[split_index:]

print("Training samples:", len(x_train_time))
print("Testing samples:", len(x_test_time))

Training samples: 597
Testing samples: 150


In [36]:
time_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

In [37]:
time_gb_model = Pipeline(
    steps=[
        ('preprocessor', time_preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=200,learning_rate=0.05,max_depth=3,random_state=42))])
time_gb_model.fit(x_train_time, y_train_time)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['meals_served','kitchen_staff','temperature_C',...,'day','week_of_year', 'is_weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This s

In [38]:
y_pred_time_gb = time_gb_model.predict(x_test_time)

In [39]:
mae_time_gb = mean_absolute_error(y_test_time, y_pred_time_gb)
rmse_time_gb = np.sqrt(mean_squared_error(y_test_time, y_pred_time_gb))
r2_time_gb = r2_score(y_test_time, y_pred_time_gb)

print("Time-Based Gradient Boosting Results")
print("="*60)
print(f"MAE  : {mae_time_gb:.3f}")
print(f"RMSE : {rmse_time_gb:.3f}")
print(f"R2 Score : {r2_time_gb:.3f}")

Time-Based Gradient Boosting Results
MAE  : 5.508
RMSE : 9.296
R2 Score : 0.887


In [40]:
time_gb_train_pred = time_gb_model.predict(x_train_time)
time_gb_train_r2 = r2_score(y_train_time, time_gb_train_pred)
print("Time-Based Gradient Boosting")
print(f"Training R2 : {time_gb_train_r2:.3f}")
print(f"Testing R2  : {r2_time_gb:.3f}")
print(f"R2 Gap      : {time_gb_train_r2 - r2_time_gb:.3f}")

Time-Based Gradient Boosting
Training R2 : 0.986
Testing R2  : 0.887
R2 Gap      : 0.099


In [41]:
time_lr_model = Pipeline(
    steps=[
        ('preprocessor', linear_preprocessor),
        ('model', LinearRegression())])
time_lr_model.fit(x_train_time, y_train_time)
y_pred_time_lr = time_lr_model.predict(x_test_time)

In [42]:
mae_time_lr = mean_absolute_error(y_test_time, y_pred_time_lr)
rmse_time_lr = np.sqrt(mean_squared_error(y_test_time, y_pred_time_lr))
r2_time_lr = r2_score(y_test_time, y_pred_time_lr)
print("Time-Based Linear Regression")
print("="*30)
print(f"MAE  : {mae_time_lr:.3f}")
print(f"RMSE : {rmse_time_lr:.3f}")
print(f"R2 Score : {r2_time_lr:.3f}")

Time-Based Linear Regression
MAE  : 5.222
RMSE : 7.863
R2 Score : 0.919


In [43]:
time_rf_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1))])
time_rf_model.fit(x_train_time, y_train_time)
y_pred_time_rf = time_rf_model.predict(x_test_time)

In [44]:
mae_time_rf = mean_absolute_error(y_test_time, y_pred_time_rf)
rmse_time_rf = np.sqrt(mean_squared_error(y_test_time, y_pred_time_rf))
r2_time_rf = r2_score(y_test_time, y_pred_time_rf)

print("Time-Based Random Forest")
print("="*30)
print(f"MAE  : {mae_time_rf:.3f}")
print(f"RMSE : {rmse_time_rf:.3f}")
print(f"R2 Score : {r2_time_rf:.3f}")

Time-Based Random Forest
MAE  : 5.615
RMSE : 8.307
R2 Score : 0.910


In [45]:
time_results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Gradient Boosting'
    ],
    'MAE': [
        mae_time_lr,
        mae_time_rf,
        mae_time_gb
    ],
    'RMSE': [
        rmse_time_lr,
        rmse_time_rf,
        rmse_time_gb
    ],
    'R2 Score': [
        r2_time_lr,
        r2_time_rf,
        r2_time_gb
    ]
})

time_results = time_results.sort_values(by='MAE')
time_results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,5.221525,7.862775,0.919304
2,Gradient Boosting,5.507568,9.295817,0.887209
1,Random Forest,5.615388,8.307032,0.909928


In [46]:
x = time_df.drop(columns=['food_waste_kg'])
y = time_df['food_waste_kg']

print("Data ready for final training.")
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")

Data ready for final training.
x shape: (747, 14)
y shape: (747,)


In [47]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

In [48]:
lr_model = Pipeline(
    steps=[
        ('preprocessor', linear_preprocessor),
        ('model', LinearRegression())])

In [49]:
lr_model.fit(x, y)
print("Final model trained successfully.")

Final model trained successfully.


In [50]:
joblib.dump(lr_model, 'final_food_waste_model.pkl')
print("Model saved successfully.")

Model saved successfully.


In [51]:
lr_model = joblib.load('final_food_waste_model.pkl')
print("Model loaded successfully.")

Model loaded successfully.


In [52]:
sample = x.iloc[[0]]
prediction = lr_model.predict(sample)
print(f"Predicted food waste: {prediction[0]:.3f} kg")

Predicted food waste: 42.255 kg
